### Import libraries

In [ ]:
from datetime import datetime, timedelta
import requests
import os
from pathlib import Path
import pandas as pd
import numpy as np
import glob
import gzip

### Global parameters

In [ ]:
RADIUS_EARTH = 6371
RADIUS_IONOSPHERE = RADIUS_EARTH + 350

IMPORTANT_COLUMNS = ['Azimuth', 'Elevation', 'Phi60_Sig1', 'Latitude', 'Longitude']
COLUMN_NAMES = ['WN', 'TOW', 'SVID', 'RxState', 'Azimuth', 'Elevation', 'AvgSig1_C/N0', 'S4_Sig1', 'Cor_S4_Sig1',
                'Phi01_Sig1', 'Phi03_Sig1', 'Phi10_Sig1', 'Phi30_Sig1', 'Phi60_Sig1', 'AvgCCD_Sig1', 'SigmaCCD_Sig1',
                'TEC_TOW-45s', 'dTEC_TOW-60_to_TOW-45', 'TEC_TOW-30s', 'dTEC_TOW-45_to_TOW-30', 'TEC_TOW-15s',
                'dTEC_TOW-30_to_TOW-15', 'TEC_TOW', 'dTEC_TOW-15_to_TOW', 'Sig1_lock_time', 'sbf2ismr',
                'Lock_time_2nd_freq', 'Avg_C/N0_2nd_freq', 'SI_ind_Sig1', 'SI_ind_Sig1_numerator', 'p_Sig1',
                'AvgSig2_C/N0', 'S4_Sig2', 'Cor_S4_Sig2', 'Phi01_Sig2', 'Phi03_Sig2', 'Phi10_Sig2', 'Phi30_Sig2',
                'Phi60_Sig2', 'AvgCCD_Sig2', 'SigmaCCD_Sig2', 'Sig2_lock_time', 'SI_ind_Sig2', 'SI_ind_Sig2_numerator',
                'p_Sig2', 'AvgSig3_C/N0', 'S4_Sig3', 'Cor_S4_Sig3', 'Phi01_Sig3', 'Phi03_Sig3', 'Phi10_Sig3',
                'Phi30_Sig3', 'Phi60_Sig3', 'AvgCCD_Sig3', 'SigmaCCD_Sig3', 'Sig3_lock_time', 'SI_ind_Sig3',
                'SI_ind_Sig3_numerator', 'p_Sig3', 'T_Sig1', 'T_Sig2', 'T_Sig3']

SELECTED_STATIONS = ['arc', 'arv', 'chu', 'cor', 'fsi', 'fsm', 'gjo', 'mcm', 'rab', 'ran', 'sac']
STATION_ABBREVIATIONS = ['arc', 'arv', 'cbb', 'chu', 'cor', 'daw', 'eur', 'mcm', 'fsi', 'fsm', 'gil', 'gjo', 'gri',
                         'hal', 'iqa', 'kug', 'edm', 'pon', 'qik', 'rab', 'ran', 'rep', 'res', 'sac', 'san', 'tal']
STATION_LATITUDES = [73.004093, 61.097941, 69.101929, 58.759279, 64.188201, 64.049559, 79.990089, 56.649535, 61.756554,
                     60.026095, 56.376600, 68.632630, 76.423281, 68.767279, 63.737377, 67.817781, 53.350818, 72.693166,
                     67.559326, 58.226935, 62.824700, 66.523589, 74.746627, 71.99063, 56.536360, 69.540923]
STATION_LONGITUDES = [274.973959, 265.928533, 254.884829, 265.913402, 276.650145, 220.888817, 274.097557, 248.779728,
                      238.771946, 248.067109, 265.356197, 264.151719, 277.096506, 278.743539, 291.459735, 244.865276,
                      247.026160, 282.044956, 295.966340, 256.322945, 267.885291, 273.768972, 264.997469, 234.739381,
                      280.768771, 266.443335]

### Defined functions

In [ ]:
def week_seconds_to_utc(gps_week, gps_seconds, leap_seconds):
    """
    Convert GPS week and seconds to UTC datetime.

    Given the GPS week, GPS seconds, and leap seconds, this function calculates
    the corresponding UTC datetime.

    Parameters:
        gps_week (int): The GPS week number.
        gps_seconds (int): The number of seconds within the GPS week.
        leap_seconds (int): The number of leap seconds at the time of conversion.

    Returns:
        str: The UTC datetime string in the format "%Y-%m-%d %H:%M:%S".
    """
    datetime_format = "%Y-%m-%d %H:%M:%S"
    epoch = datetime.strptime("1980-01-06 00:00:00", datetime_format)
    elapsed = timedelta(days=(gps_week * 7), seconds=(gps_seconds + leap_seconds))
    
    return datetime.strftime(epoch + elapsed, datetime_format)   



def load_bins_data():
    """
    Load bin positions and prepare arrays for quick bin searching.

    This function loads the bin positions from the "Bins_equidistant.txt" file and prepares
    the necessary arrays for quick bin searching. Next, the function extracts the unique latitude values.
    For each unique latitude value, an associated array of longitude values is created. This allows
    faster searching of bins based on latitude.

    Returns:
        tuple: A tuple containing the following elements:
            - bins (Pandas DataFrame): Bin positions as a Pandas DataFrame.
            - unique_lat (NumPy array): Array of unique latitude values.
            - associated_lon (list): List of arrays containing associated longitude values for each unique latitude.
    """
    bins_column_names = ['Latitude_bins', 'Longitude_bins']
    bins_file = Path('Bins_equidistant.txt')
    with open(bins_file, 'r') as f:
        bins = pd.read_table(f, sep='\t', names=bins_column_names, skipinitialspace=True)

    # prepare arrays for quick bin searching
    unique_lat = np.unique(np.array((bins['Latitude_bins'])))
    associated_lon = []
    for i in range(len(unique_lat)):
        associated_lon.append([])
    for i in range(len(bins)):
        idx_bin = list(unique_lat).index(bins.iloc[i, 0])
        associated_lon[idx_bin].append(bins.iloc[i, 1])

    return bins, unique_lat, associated_lon



def sort_data_into_bins(df, bins, unique_lat, associated_lon):
    """
    Sorts data into bins based on Ionospheric Pierce Point (IPP) and returns the updated DataFrame.

    This function calculates the IPP (Ionospheric Pierce Point) coordinates for each data point in the
    provided DataFrame and determines the closest bin for each IPP.

    Parameters:
        df (Pandas DataFrame): The DataFrame containing the data to be sorted into bins.
        bins (Pandas DataFrame): The bin locations as a DataFrame.
        unique_lat (NumPy array): An array of unique latitude values.
        associated_lon (list): A list of arrays containing the associated longitude values for each unique latitude.

    Returns:
        Pandas DataFrame: The updated DataFrame with the following additional columns:
            - 'Latitude_IPP': Latitude of the Ionospheric Pierce Point (IPP) for each data point.
            - 'Longitude_IPP': Longitude of the Ionospheric Pierce Point (IPP) for each data point.
            - 'Latitude_bin': Latitude of the closest bin for each data point.
            - 'Longitude_bin': Longitude of the closest bin for each data point.
            - 'Bin_index': Index of the closest bin for each data point.
    """
    print("Sorting data into bins based on Ionospheric Pierce Point (IPP)...")
    # calculate ionospheric pierce point
    df['Latitude_IPP'], df['Longitude_IPP'] = calc_ipp_geometric(station_lat=np.array(np.radians(df['Latitude'])),
                                                                 station_lon=np.array(np.radians(df['Longitude'])),
                                                                 azimuth=np.array(np.radians(df['Azimuth'])),
                                                                 elevation=np.array(np.radians(df['Elevation'])))
    # calculate closest bins
    applied_df = df.apply(lambda row: calc_close_bin_faster(np.array(bins['Latitude_bins']),
                                                            unique_lat,
                                                            np.array(associated_lon, dtype=object),
                                                            row['Latitude_IPP'],
                                                            row['Longitude_IPP']), axis=1, result_type='expand')

    applied_df = applied_df.rename(columns={0: 'Latitude_bin', 1: 'Longitude_bin', 2: 'Bin_index'})
    df['Latitude_bin'] = applied_df['Latitude_bin']
    df['Longitude_bin'] = applied_df['Longitude_bin']
    df['Bin_index'] = applied_df['Bin_index']

    return df



def calc_ipp_geometric(station_lat, station_lon, azimuth, elevation, radius_earth=RADIUS_EARTH,
                       radius_ionosphere=RADIUS_IONOSPHERE):
    """
    Calculate the Ionospheric Pierce Point (IPP) coordinates using geometric method.

    Given the station's latitude, longitude, azimuth, and elevation, along with the
    Earth's radius and ionosphere's radius, this function computes the IPP latitude
    and longitude coordinates using the geometric method.

    Parameters:
        station_lat (NumPy array): Latitude of the station in radians.
        station_lon (NumPy array): Longitude of the station in radians.
        azimuth (NumPy array): Azimuth angle in radians.
        elevation (NumPy array): Elevation angle in radians.
        radius_earth (float, optional): Radius of the Earth in kilometers. Default is RADIUS_EARTH.
        radius_ionosphere (float, optional): Radius of the ionosphere in kilometers. Default is RADIUS_IONOSPHERE.

    Returns:
        tuple: A tuple containing the IPP latitude and longitude coordinates in degrees.
    """
    psi = (np.pi/2) - elevation - np.arcsin((radius_earth / radius_ionosphere) * np.cos(elevation))
    ipp_lat = np.arcsin((np.sin(station_lat) * np.cos(psi)) + (np.cos(station_lat) * np.sin(psi) * np.cos(azimuth)))
    ipp_lon = station_lon + np.arcsin((np.sin(psi) * np.sin(azimuth)) / np.cos(ipp_lat))

    return np.degrees(ipp_lat), np.degrees(ipp_lon)



def calc_close_bin_faster(bins_lat_not_unique, bins_lat_unique, bins_lon_associated, lat, lon):
    """
    Find the nearest bin faster by first searching for the nearest latitude, then longitude.

    Given a list of non-unique bin latitudes, unique bin latitudes, associated bin longitudes,
    and a target latitude and longitude, this function first identifies the closest latitude
    to the target and then searches for the nearest bin based on longitude. Finally, it returns
    the latitude, longitude, and global index of the nearest bin.

    Parameters:
        bins_lat_not_unique (NumPy array): List of non-unique latitude coordinates for the bins.
        bins_lat_unique (NumPy array): List of unique latitude coordinates for the bins.
        bins_lon_associated (NumPy array): List of lists containing associated longitude coordinates
                                           for each bin latitude.
        lat (float): Target latitude in degrees.
        lon (float): Target longitude in degrees.

    Returns:
        tuple: A tuple containing the latitude, longitude, and global index of the nearest bin.
    """
    closest_lat = np.argmin(np.abs(bins_lat_unique - lat))
    bins_lat_temp = np.array([bins_lat_unique[closest_lat]] * len(bins_lon_associated[closest_lat]))
    bins_lat, bins_lon, min_index = calc_near_bin(bins_lat_temp, np.array(bins_lon_associated[closest_lat]), lat, lon)
    global_min_index = np.where(bins_lat_not_unique == bins_lat)[0][0]
    
    return bins_lat, bins_lon, global_min_index + min_index



def sort_data_into_bins(df, bins, unique_lat, associated_lon):
    """
    Sorts data into bins based on Ionospheric Pierce Point (IPP) and returns the updated DataFrame.

    This function calculates the IPP (Ionospheric Pierce Point) coordinates for each data point in the
    provided DataFrame and determines the closest bin for each IPP.

    Parameters:
        df (Pandas DataFrame): The DataFrame containing the data to be sorted into bins.
        bins (Pandas DataFrame): The bin locations as a DataFrame.
        unique_lat (NumPy array): An array of unique latitude values.
        associated_lon (list): A list of arrays containing the associated longitude values for each unique latitude.

    Returns:
        Pandas DataFrame: The updated DataFrame with the following additional columns:
            - 'Latitude_IPP': Latitude of the Ionospheric Pierce Point (IPP) for each data point.
            - 'Longitude_IPP': Longitude of the Ionospheric Pierce Point (IPP) for each data point.
            - 'Latitude_bin': Latitude of the closest bin for each data point.
            - 'Longitude_bin': Longitude of the closest bin for each data point.
            - 'Bin_index': Index of the closest bin for each data point.
    """
    print("Sorting data into bins based on Ionospheric Pierce Point (IPP)...")
    # calculate ionospheric pierce point
    df['Latitude_IPP'], df['Longitude_IPP'] = calc_ipp_geometric(station_lat=np.array(np.radians(df['Latitude'])),
                                                                 station_lon=np.array(np.radians(df['Longitude'])),
                                                                 azimuth=np.array(np.radians(df['Azimuth'])),
                                                                 elevation=np.array(np.radians(df['Elevation'])))
    # calculate closest bins
    applied_df = df.apply(lambda row: calc_close_bin_faster(np.array(bins['Latitude_bins']),
                                                            unique_lat,
                                                            np.array(associated_lon, dtype=object),
                                                            row['Latitude_IPP'],
                                                            row['Longitude_IPP']), axis=1, result_type='expand')

    applied_df = applied_df.rename(columns={0: 'Latitude_bin', 1: 'Longitude_bin', 2: 'Bin_index'})
    df['Latitude_bin'] = applied_df['Latitude_bin']
    df['Longitude_bin'] = applied_df['Longitude_bin']
    df['Bin_index'] = applied_df['Bin_index']

    return df



def calc_near_bin(bins_lat, bins_lon, lat, lon):
    """
    Search for the nearest bin based on latitude and longitude coordinates.

    Given a list of bin latitudes, bin longitudes, and a target latitude and longitude,
    this function calculates the distances between the target coordinates and each bin
    using the `calc_arc_length_pre` function. It then returns the latitude, longitude, and
    index of the nearest bin.

    Parameters:
        bins_lat (NumPy array): List of latitude coordinates for the bins.
        bins_lon (NumPy array): List of longitude coordinates for the bins.
        lat (float): Target latitude in degrees.
        lon (float): Target longitude in degrees.

    Returns:
        tuple: A tuple containing the latitude, longitude, and index of the nearest bin.
    """
    distances = calc_arc_length_pre(bins_lat, lat, bins_lon, lon)
    min_index = np.argmin(distances)
    
    return bins_lat[min_index], bins_lon[min_index], min_index



def calc_arc_length_pre(lat1, lat2, lon1, lon2):
    """
    Calculate the arc length between two points on Earth in degrees.

    Given the latitude and longitude coordinates of two points on Earth, this function
    calculates the arc length between the points.

    Parameters:
        lat1 (float): Latitude of the first point in degrees.
        lat2 (float): Latitude of the second point in degrees.
        lon1 (float): Longitude of the first point in degrees.
        lon2 (float): Longitude of the second point in degrees.

    Returns:
        float: The arc length between the two points on Earth in degrees.
    """
    lat1, lat2, lon1, lon2 = np.radians(lat1), np.radians(lat2), np.radians(lon1), np.radians(lon2)
    numerator = np.sqrt((np.cos(lat2) * np.sin(lon2-lon1))**2 +
                        (np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(lon2-lon1))**2)
    denominator = np.sin(lat1) * np.sin(lat2) + np.cos(lat1) * np.cos(lat2) * np.cos(lon2-lon1)
    c = np.arctan(numerator / denominator)
    c[c < 0] = np.pi + c[c < 0]
    
    return np.degrees(c)



def make_measurements(df):
    """
    Process measurements from a DataFrame and return aggregated values.

    The function takes a DataFrame `df` and performs the following steps:
        - Groups the data by bin index and calculates the mean for each group.
        - Constructs a new DataFrame with aggregated values.

    Parameters:
        df (pandas.DataFrame): Input DataFrame with measurements.

    Returns:
        pandas.DataFrame: DataFrame with aggregated measurements.
    """
    print("Processing measurements...")
    measurements_df = pd.DataFrame()

    bin_unique = np.unique(np.array(df['Bin_index']), return_counts=True)
    for i in range(len(bin_unique[0])):
        df_part = df[df['Bin_index'] == bin_unique[0][i]]
        df_part.index = pd.to_datetime(df_part.index, utc=True)
        df_part = df_part.groupby(df_part.index).mean()

        # fill measurements DataFrame
        bin_index = [int(bin_unique[0][i])] * len(df_part)
        measurements_tmp_df = pd.DataFrame({'Phi60_Sig1': np.array(df_part['Phi60_Sig1']), 'Bin_index': bin_index,
                                            'Time': df_part.index})
        measurements_tmp_df['Time'] = measurements_tmp_df['Time'].map(lambda x: x.isoformat())
        measurements_df = pd.concat([measurements_df, measurements_tmp_df])

    print(f"Number of processed measurements: {len(measurements_df)}")

    return measurements_df

In [ ]:
# SELECT year, month, day, hour
t = datetime(2021, 10, 12, 6)

In [ ]:
day_of_year = t.timetuple().tm_yday
year = t.year
hour = t.hour

number_of_not_loaded_stations = 0
alphabet = [chr(letter) for letter in range(ord('a'), ord('z') + 1)]

for receiver in SELECTED_STATIONS:
    url = f'http://chain.physics.unb.ca/data/gps/ismr/{year}/{day_of_year:03}/{hour:02}/{receiver}' \
            f'c{str(year)[2:]}{day_of_year:03}{alphabet[hour]}.ismr.gz'
    response = requests.get(url)
    if response.status_code == 200:
        file_name = url.split('/')[-1]
        with open(str(os.path.join('tmp', file_name)), 'wb') as f:
            f.write(response.content)
            print(f"File '{file_name}' written successfully.")
    else:
        number_of_not_loaded_stations += 1
        print(f"URL '{url}' does not exist.")

if number_of_not_loaded_stations == len(SELECTED_STATIONS):
    print(f'{test_dir_url} does not contain the data we need.')
        
else:
    print(f'Data downloaded.')

In [ ]:
directory = Path('tmp/*.gz')

df = pd.DataFrame(columns=COLUMN_NAMES)
for file in glob.iglob(str(directory)):
    station = Path(file).stem[0:3]
    print(f"Processing file: {file}")
    with open(file, 'rb') as f:
        gzip_f = gzip.GzipFile(fileobj=f)
        data = pd.read_table(gzip_f, sep=',', names=COLUMN_NAMES, skipinitialspace=True)
            
    idx_station = STATION_ABBREVIATIONS.index(station)
    data['Station'] = [station] * len(data)
    data['Latitude'] = [STATION_LATITUDES[idx_station]] * len(data)
    data['Longitude'] = [STATION_LONGITUDES[idx_station]] * len(data)
    data.index = data.apply(lambda row: week_seconds_to_utc(row['WN'], row['TOW'], 0), axis=1)
    df = pd.concat([df, data])

    # we don't need the file anymore, so to save space we delete it
    os.remove(file)

# apply constraints and delete NaN values
df = df.loc[(df['Elevation'] >= 20)]
df = df.loc[(df['Phi60_Sig1'] < 8)]
df.dropna(subset=['Azimuth', 'Elevation', 'Phi60_Sig1'], inplace=True)

# cleans the data frame from unnecessary columns and retypes the necessary columns
df = df[IMPORTANT_COLUMNS]
df.sort_index(inplace=True)
df['Azimuth'] = df['Azimuth'].astype(float)
df['Elevation'] = df['Elevation'].astype(float)

In [ ]:
bins, unique_lat, associated_lon = load_bins_data()
df = sort_data_into_bins(df, bins, unique_lat, associated_lon)

measurements_df = make_measurements(df)

In [ ]:
measurements_df.sort_values(["Time", "Bin_index"]).reset_index(drop=True)